<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/clean2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === Section 0: Install dependencies ===
!pip install -q openai google-generativeai

In [ ]:
  # === Section 0: Mount Drive + Imports + Config ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, time, re, zipfile, random, threading
import concurrent.futures as cf
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

DRIVE_BASE = '/content/drive/MyDrive/ThaiScamCall'
OUT_DIR    = f'{DRIVE_BASE}/clean'
ZIP_AUDIO  = f'{DRIVE_BASE}/mp3_15s.zip'
EXT_AUDIO  = '/content/data_15s'
os.makedirs(OUT_DIR, exist_ok=True)

gpt = OpenAI(
       base_url='https://openrouter.ai/api/v1',
       api_key=userdata.get('OPENROUTER_KEY')
   )

# ===== Config =====
MODE          = 'all'   # 🔧 'sample'(300,~$0.05) ดูก่อน | 'all'(21K,~$2-3)
SAMPLE_N      = 300
GPT_WORKERS   = 20        # 🔧 ขนาน GPT (มี retry กัน rate limit)
PROMPT_VER    = 'v5'       # 🔧 แก้ prompt → เปลี่ยนเลข → checkpoint รีเฟรช

REALISTIC_MIN = 3          # 🔧 realistic < ค่านี้ = ลบ (scenario ไม่สมจริง)

FORMAL_MIN     = 5
GEMINI_WORKERS= 20
GEMINI_TOPN = 100000   # 🔧 ใส่ใหญ่ๆ → เอาทุกตัวที่เป็น formal+keep

print('✅ Setup OK')
print(f'   MODE={MODE} | GPT workers={GPT_WORKERS} | prompt={PROMPT_VER}')
print(f'   realistic<{REALISTIC_MIN} → remove | Gemini top-{GEMINI_TOPN}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup OK
   MODE=all | GPT workers=20 | prompt=v5
   realistic<3 → remove | Gemini top-100000


In [ ]:
# === Section 1: Load dataset_cleaned.jsonl ===
from google.colab import files
print('เลือก dataset_cleaned.jsonl ...')
up = files.upload()
fname = list(up.keys())[0]

data = []
for line in up[fname].decode('utf-8').splitlines():
    if line.strip():
        data.append(json.loads(line))

for i, d in enumerate(data):
    d['line']    = i + 1
    d['conv_id'] = f'conv_{i+1:05d}_label{d["label"]}'   # ⭐ ใช้ label เดิม (ตรงกับชื่อไฟล์เสียง)

def full_text(d):
    return ' '.join(t['text'] for t in d['turns'])

print(f'Total: {len(data)} บท')
print(pd.Series([d['label'] for d in data]).value_counts()
      .rename({0:'not_scam', 1:'scam'}))

เลือก dataset_cleaned.jsonl ...


Saving dataset_cleaned.jsonl to dataset_cleaned (1).jsonl
Total: 21352 บท
not_scam    11212
scam        10140
Name: count, dtype: int64


In [ ]:
# === Section 2: ChatGPT check function (label + realistic + formal_score) ===
SYSTEM = """คุณตรวจสอบบทสนทนาโทรศัพท์ (ฝึก AI ตรวจ scam call ไทย) จาก transcript
label เดิม: 0=ปกติ(not_scam), 1=scam — เชื่อ label เดิมไว้ก่อน

ตอบ JSON:
{
  "true_label": 0 หรือ 1,    // label ที่ถูก (ไม่มั่นใจ = label เดิม)
  "realistic": 0-10,         // ฉากนี้เกิดได้จริงในชีวิตจริงไหม (ไม่เกี่ยวว่า scam/ไม่)
  "formal_score": 0-10,      // ทางการ: ตำรวจ/ธนาคาร/หน่วยงาน/บริษัท=สูง | เพื่อน/คุยเล่น=ต่ำ
  "action": "keep"/"relabel"/"remove",
  "reason": "สั้นๆ"
}

⚠️ relabel asymmetric:
- 0→1 (เติม scam ตกหล่น): ทำได้ถ้าเจอ pattern scam **ชัดเจน** (ดูตัวอย่างล่าง)
- 1→0 (ถอด scam): ห้าม ยกเว้นบทคุยเล่นล้วนๆ ไม่มีเสนอเงิน/บริการ/ขอข้อมูล

🎯 SCAM ชัด (relabel 0→1 ได้):
- เพื่อน/ญาติ "เปลี่ยนเบอร์ใหม่" + ยืมเงิน = scam ไทยคลาสสิก
- หน่วยงาน (ตำรวจ/สรรพากร/ธนาคาร/ปปง) ขู่/อายัด/คดี/ฟอกเงิน → โอนเงิน
- ขอให้เหยื่อบอก OTP/รหัส/เลขบัตร (auditor เชื่อ → เหยื่อบอก)
- กดลิงก์/ติดตั้งแอป/USSD เพื่อยืนยัน
- สินเชื่อ-เงินด่วน อนุมัติง่าย/ไม่เช็คเครดิต
- ลงทุนผลตอบแทนสูง/งานออนไลน์รายได้ดีเกินจริง
- เงินคืนภาษี/รางวัล ที่ต้องกดรหัส/โอนค่าธรรมเนียม

✅ ปกติ NOT scam (ห้าม relabel เป็น scam) — ระวังพลาด:
- "ระบบ**ส่ง** OTP/รหัส**ให้**ผู้ใช้" เช่น "รหัสของคุณคือ 123456 ห้ามเปิดเผย",
  "ใช้รหัสยืนยัน 123456 เพื่อรีเซ็ต..." = ปกติ (ระบบบอกเหยื่อ ไม่ใช่ขอ)
- โทรขายโปรมือถือ/เน็ต (dtac/AIS/true) ที่ไม่ขอเงิน-ข้อมูลลับ = ปกติ
- ขายประกันสุขภาพ/ชีวิตปกติ (มีเบี้ย, ให้คิดดู) = ปกติ
- เพื่อนยืมเงินเล็กน้อย (50บาท/100บาท/ค่าข้าว) ที่คุยปกติ = ปกติ
- พ่อแม่/ลูก/ครอบครัวจริง คุยเรื่องเงินในบ้าน = ปกติ

action=remove = บทเสีย: ว่าง / มีแต่ breathing noise / instruction หลุด ("แทนคำว่า") /
  สั้นเกินตัดสินไม่ได้ / รายการคำ ไม่เป็นบทสนทนา

realistic ต่ำ = ฉากไม่น่าเกิดจริง (ธนาคารโทรขอ update บัตร) — ไม่เกี่ยว scam หรือไม่"""

def gpt_check(d):
    txt = '\n'.join(f"{t['speaker']}: {t['text']}" for t in d['turns'])
    for attempt in range(4):
        try:
            r = gpt.chat.completions.create(
                model='openai/gpt-4o-mini',
                messages=[{'role':'system','content':SYSTEM},
                          {'role':'user','content':f'label เดิม={d["label"]}\nบทสนทนา:\n{txt}'}],
                temperature=0, max_tokens=130,
                response_format={'type':'json_object'})
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            if attempt == 3:
                return {'action':'error', 'reason':str(e)[:60]}
            time.sleep(2 ** attempt)

print('Test:', gpt_check(data[0]))

Test: {'true_label': 1, 'realistic': 3, 'formal_score': 8, 'action': 'keep', 'reason': 'ขอรหัส OTP ชัดเจนว่าเป็น scam'}


In [ ]:
# === Section 3: Run GPT (parallel + checkpoint + resume errors) ===
target = (random.Random(42).sample(data, min(SAMPLE_N, len(data)))
          if MODE == 'sample' else data)

CKPT = f'{OUT_DIR}/gpt_ckpt_{MODE}_{PROMPT_VER}.jsonl'
done = {}
if os.path.exists(CKPT):
    for ln in open(CKPT, encoding='utf-8'):
        if ln.strip():
            o = json.loads(ln); done[o['conv_id']] = o

# retry: ตัวที่ยังไม่ทำ หรือเคย error → ทำใหม่
todo = [d for d in target
        if d['conv_id'] not in done or done[d['conv_id']].get('action') == 'error']
print(f'MODE={MODE} | target={len(target)} | done={len(done)} | todo={len(todo)} '
      f'| ~{len(todo)/GPT_WORKERS*0.9/60:.0f} นาที')

lock = threading.Lock()
ckf  = open(CKPT, 'a', encoding='utf-8')
def work(d):
    r = gpt_check(d)
    rec = {'conv_id':d['conv_id'],
           'true_label':r.get('true_label', d['label']),
           'realistic':r.get('realistic', 5),
           'formal_score':r.get('formal_score', 0),
           'action':r.get('action', 'keep'),
           'reason':r.get('reason', '')}
    with lock:
        ckf.write(json.dumps(rec, ensure_ascii=False) + '\n'); ckf.flush()
    return rec
with cf.ThreadPoolExecutor(GPT_WORKERS) as ex:
    list(tqdm(ex.map(work, todo), total=len(todo), desc='GPT'))
ckf.close()

# merge checkpoint → d objects
done = {}
for ln in open(CKPT, encoding='utf-8'):
    if ln.strip():
        o = json.loads(ln); done[o['conv_id']] = o
for d in target:
    o = done.get(d['conv_id'], {})
    d['gpt_label']    = o.get('true_label', d['label'])
    d['realistic']    = o.get('realistic', 5)
    d['formal_score'] = o.get('formal_score', 0)
    d['action']       = o.get('action', 'keep')
    d['reason']       = o.get('reason', '')

# สรุป
acts = Counter(d['action'] for d in target)
up   = sum(d['action']=='relabel' and d['label']==0 and d['gpt_label']==1 for d in target)
down = sum(d['action']=='relabel' and d['label']==1 and d['gpt_label']==0 for d in target)
low  = sum(d['realistic'] < REALISTIC_MIN for d in target)
print('\n=== Action ===')
for a,c in acts.items(): print(f'  {a}: {c} ({c/len(target)*100:.1f}%)')
print(f'  relabel 0→1: {up} | 1→0: {down}')
print(f'  realistic<{REALISTIC_MIN} (จะลบ): {low} ({low/len(target)*100:.1f}%)')
print(f'  errors: {acts.get("error",0)}')

MODE=all | target=21352 | done=6623 | todo=16734 | ~13 นาที


GPT:   0%|          | 0/16734 [00:00<?, ?it/s]


=== Action ===
  keep: 18423 (86.3%)
  remove: 1844 (8.6%)
  relabel: 1085 (5.1%)
  relabel 0→1: 993 | 1→0: 34
  realistic<3 (จะลบ): 482 (2.3%)
  errors: 0


In [ ]:
# === Section 4: Inspect results + save QA CSV ===
df = pd.DataFrame([{
    'conv_id':d['conv_id'], 'label':d['label'], 'gpt_label':d['gpt_label'],
    'realistic':d['realistic'], 'formal_score':d['formal_score'],
    'action':d['action'], 'reason':d['reason'], 'text':full_text(d)[:90],
} for d in target])

print('=== RELABEL (label แก้) ===')
display(df[df.action=='relabel'][['conv_id','label','gpt_label','reason','text']].head(20))
print('=== REMOVE (บทเสีย) ===')
display(df[df.action=='remove'][['conv_id','reason','text']].head(10))
print(f'=== Realistic < {REALISTIC_MIN} (จะลบ) ===')
display(df[df.realistic<REALISTIC_MIN][['conv_id','label','realistic','reason','text']].head(10))
print('=== Formal score สูงสุด (จะส่ง Gemini) ===')
display(df.sort_values('formal_score', ascending=False)
        [['conv_id','label','formal_score','reason','text']].head(10))

# QA CSV — utf-8-sig เปิด Excel อ่าน Thai ออก, ใส่ T/F เอง
rl = df[df.action=='relabel'].copy()
rl['full_text'] = [full_text(d) for d in target if d['action']=='relabel']
rl['correct(T/F)'] = ''
rl[['conv_id','label','gpt_label','reason','full_text','correct(T/F)']].to_csv(
    f'{OUT_DIR}/relabels_review_{MODE}.csv', index=False, encoding='utf-8-sig')
print(f'\n💾 saved relabels_review_{MODE}.csv ({len(rl)} ตัว) — verify T/F ส่ง')

=== RELABEL (label แก้) ===


,conv_id,label,gpt_label,reason,text
34,conv_00035_label0,0,1,ขอ OTP จากผู้ใช้ ซึ่งเป็นลักษณะ scam,พี่จ๋า สวัสดีครับ นี่ธนาคารนะครับ ค่ะ มีอะไรเห...
53,conv_00054_label0,0,1,เพื่อนขอเงิน 500 บาท ถือเป็น scam classic,"น้องครับ สบายดีไหม นี่, มันคือ อ้น เองนะ ได้เบ..."
123,conv_00124_label0,0,1,ขอ OTP เพื่อยืนยันรายการชัดเจนว่าเป็น scam,ขอโทษครับ คุณ ชื่อ ดิฉันจากธนาคารนะครับ ต้องกา...
159,conv_00160_label0,0,1,ขอให้เหยื่อยืนยัน OTP ชัดเจน,ขออนุญาตครับ คุณลูกค้า นี่ธนาคารนะครับ เรามีธุ...
167,conv_00168_label0,0,1,เพื่อนขอเงินยืมชัดเจน,พี่ครับ คือกูมีเบอร์ใหม่อ่ะ เน็ตมันหมดเลยต้องเ...
172,conv_00173_label0,0,1,เพื่อนยืมเงินและสัญญาจะคืนทันที เป็น pattern s...,สวัสดีค่ะ แก เป็นไงบ้างช่วงนี้ ก็ดีจ้า แกเองล่...
201,conv_00202_label1,1,0,บทสนทนาฟังดูเหมือนคุยเล่น ไม่มีการขอข้อมูลหรือ...,"เห้, สวัสดีครับ คุณแอนนาใช่ไหม ผมเจคจากอเมริกา..."
249,conv_00250_label0,0,1,เพื่อนขอเงิน 500 บาทหลังเปลี่ยนเบอร์ใหม่ ชัดเจ...,เฮ้ย นี่กูเองนะ นัท เพิ่งเปลี่ยนเบอร์ใหม่อ่ะ ค...
263,conv_00264_label0,0,1,เสนอวงเงินบัตรเครดิตโดยไม่ขอข้อมูลส่วนตัวชัดเจน,หวัดดีครับ คุณจะรับรู้ว่าเรามีโปรโมชั่นใหม่จาก...
288,conv_00289_label0,0,1,ขอ OTP จากเหยื่อ เป็น scam ชัดเจน,พี่จ๋า สวัสดีครับ ผมจากธนาคารนะครับ สวัสดีครับ...


=== REMOVE (บทเสีย) ===


,conv_id,reason,text
1,conv_00002_label0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย อ่า ก็ คือ ไม่รู้ว่า อืม แกอยู่ไหน
4,conv_00005_label0,บทสนทนาสั้นเกินไป ไม่มีข้อมูลที่ชัดเจน,สวัสดีค่ะ เธอสบายดีมั้ย อ๋อ สบายดีจ้ะ แล้วเธอล...
7,conv_00008_label0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย แก คือ ไม่รู้ว่าทำไม
9,conv_00010_label0,บทสนทนาสั้นเกินไป ไม่มีข้อมูลเพียงพอ,ฮัลโหล มีใครอยู่ไหม
10,conv_00011_label0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย มีใครอยู่ปลายสายป่าว
27,conv_00028_label0,บทสนทนาธรรมดา ไม่มีเนื้อหาสำคัญ,เฮ้ย คุณครับ วันนี้มีประชุมกี่โมงอ่ะ อ๋อ ประชุ...
32,conv_00033_label0,บทสนทนาสั้นเกินไป ไม่มีข้อมูลที่ชัดเจน,เฮ้ย แก ฟังไม่ค่อยชัดเลย อ่ะ มีเสียงสะท้อน อืม...
37,conv_00038_label0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย แก ได้ยินป่ะ เสียงมันแปลกๆ อ่ะ อือ ไม่ค่อ...
60,conv_00061_label0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,ฮัลโหล ใครครับ
68,conv_00069_label0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,หวัดดีค่ะ สวัสดีค่ะ คุณสมิทธิ์


=== Realistic < 3 (จะลบ) ===


,conv_id,label,realistic,reason,text
1,conv_00002_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย อ่า ก็ คือ ไม่รู้ว่า อืม แกอยู่ไหน
7,conv_00008_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย แก คือ ไม่รู้ว่าทำไม
9,conv_00010_label0,0,2,บทสนทนาสั้นเกินไป ไม่มีข้อมูลเพียงพอ,ฮัลโหล มีใครอยู่ไหม
10,conv_00011_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย มีใครอยู่ปลายสายป่าว
60,conv_00061_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,ฮัลโหล ใครครับ
69,conv_00070_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย มีใครอยู่ไหม
192,conv_00193_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เออ คือ อ๋อ ไม่ได้ เขา แล้วก็คือ ทำไม มึงว่า อืม
213,conv_00214_label0,0,0,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,
217,conv_00218_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เอ่อ คือ มันมี แบบว่า ต้องการจะ อะไรนะ
247,conv_00248_label0,0,2,บทสนทนาสั้นเกินไป ไม่สามารถตัดสินได้,เฮ้ย ใครวะ


=== Formal score สูงสุด (จะส่ง Gemini) ===


,conv_id,label,formal_score,reason,text
7265,conv_07266_label0,0,10,เป็นการแจ้งรหัส OTP จากระบบธนาคาร,สวัสดีค่ะ นี่คือข้อความยืนยันตัวตนจากธนาคารของ...
12485,conv_12486_label0,0,10,เป็นการแจ้งรหัส OTP จากระบบธนาคาร,สวัสดีค่ะ นี่คือข้อความจากธนาคารของคุณ รหัส OT...
16256,conv_16257_label1,1,9,ขอ OTP จากลูกค้า ชัดเจนว่าเป็น scam,ฮัลโหล สวัสดีครับคุณลูกค้า นี่ธนาคารนะครับ เรา...
12508,conv_12509_label0,0,9,ขอ OTP เพื่อยืนยันการทำรายการ เป็น scam ชัดเจน,น้องครับ สวัสดีครับ ผมจากธนาคารนะครับ ต้องการย...
5996,conv_05997_label0,0,9,ระบบส่ง OTP ให้ผู้ใช้,รหัสยืนยันในการรีเซ็ตรหัสผ่านของคุณคือ 123456 ...
2273,conv_02274_label1,1,9,มีการขู่และขอให้โอนเงิน,สวัสดีครับ คุณผู้โชคดี นี่คือกรมสอบสวนคดีพิเศษ...
2278,conv_02279_label0,0,9,ระบบส่ง OTP ให้ผู้ใช้ ไม่ใช่ขอข้อมูล,สวัสดีค่ะ นี่คือรหัสการยืนยันสำหรับการรีเซ็ตรห...
2217,conv_02218_label0,0,9,ระบบส่ง OTP ให้ผู้ใช้ ไม่ใช่ขอข้อมูล,ท่านได้รับรหัสยืนยันตัวตนคือ 123456 กรุณาอย่าเ...
2225,conv_02226_label0,0,9,เป็นการแจ้งรหัส OTP จากระบบ ไม่ใช่การขอข้อมูล,ท่านได้รับรหัสยืนยัน OTP คือ 123456 กรุณาอย่าเ...
2234,conv_02235_label0,0,9,บทสนทนาปกติจากธนาคารเกี่ยวกับการแจ้งเตือนบัตรเ...,สวัสดีครับ คุณลูกค้า นี่คือธนาคารของเราแจ้งเตื...



💾 saved relabels_review_all.csv (1085 ตัว) — verify T/F ส่ง


In [ ]:
  # === Section 5: Apply clean → dataset_relabeled.jsonl ===
APPLY = True   # 🔧 True เมื่อ MODE='all' + ตรวจผลแล้ว

if APPLY and MODE == 'all':
    clean = []; n_rm = 0; n_rl = 0; rm_lines = set()
    for d in data:
        # ลบ: บทเสีย หรือ ฉากไม่สมจริง
        if d['action'] == 'remove' or d['realistic'] < REALISTIC_MIN:
            n_rm += 1; rm_lines.add(d['line']); continue
        lbl = d['gpt_label'] if d['action'] == 'relabel' else d['label']
        if d['action'] == 'relabel': n_rl += 1
        clean.append({'label': lbl, 'turns': d['turns']})   # schema เดิม {label,turns}

    # บันทึก jsonl ลงเครื่องก่อนค่อยย้าย
    local_jsonl = 'dataset_relabeled.jsonl'
    with open(local_jsonl, 'w', encoding='utf-8') as f:
        for d in clean:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    !cp {local_jsonl} {OUT_DIR}/{local_jsonl}

    # zip เสียงที่เหลือ (ย้ายมาทำที่ local /content/ ทั้งหมด)
    if not os.path.exists(EXT_AUDIO) or len(os.listdir(EXT_AUDIO)) < 1000:
        print('Extracting audio to local storage...')
        with zipfile.ZipFile(ZIP_AUDIO) as z: z.extractall(EXT_AUDIO)

    keep_au = [f for f in os.listdir(EXT_AUDIO) if f.endswith('.mp3')
               and int(re.search(r'conv_(\d+)_', f).group(1)) not in rm_lines]

    local_zip = 'mp3_15s_clean.zip'
    with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in tqdm(keep_au, desc='zip audio (local)'):
            zf.write(f'{EXT_AUDIO}/{fn}', arcname=fn)

    # ย้าย zip เข้า Drive ในคำสั่งเดียว
    !cp {local_zip} {DRIVE_BASE}/{local_zip}

    print(f'✅ dataset_relabeled.jsonl: {len(clean)} บท')
    print(f'✅ mp3_15s_clean.zip: {len(keep_au)} ไฟล์ ย้ายเข้า Drive แล้ว')
else:
    print('⏸️ ตั้ง MODE=all + APPLY=True ก่อนรัน')

Extracting audio to local storage...


zip audio (local):   0%|          | 0/19489 [00:00<?, ?it/s]

✅ dataset_relabeled.jsonl: 19500 บท
✅ mp3_15s_clean.zip: 19489 ไฟล์ ย้ายเข้า Drive แล้ว


In [ ]:
# === Section 6: Gemini ผ่าน OpenRouter ===
import base64
from openai import OpenAI

or_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=userdata.get('OPENROUTER_KEY')
)
GEMINI_MODEL = 'google/gemini-2.5-flash'

if not os.path.exists(EXT_AUDIO) or len(os.listdir(EXT_AUDIO)) < 1000:
    print('Extracting audio...')
    with zipfile.ZipFile(ZIP_AUDIO) as z: z.extractall(EXT_AUDIO)
print(f'Local audio: {len(os.listdir(EXT_AUDIO))}')

def voice_check(conv_id, text):
    fn = f'{EXT_AUDIO}/{conv_id}.mp3'
    if not os.path.exists(fn):
        return {'mismatch': None, 'reason': 'no audio', 'status': 'skip'}
    with open(fn, 'rb') as f:
        audio_b64 = base64.b64encode(f.read()).decode('utf-8')
    prompt_text = f"""ฟังเสียง + อ่าน transcript เทียบกัน
transcript: "{text[:300]}"
เนื้อหานี้ทางการ/จริงจัง (ธนาคาร/ตำรวจ/หน่วยงาน) น้ำเสียงควรสุภาพ-น่าเชื่อถือ
mismatch=true ถ้าเสียง: หัวเราะ/เล่นๆ/ทีเล่นทีจริง/ร่าเริงเกิน/อารมณ์ไม่เข้าเนื้อหา
ตอบ JSON: {{"mismatch": true/false, "reason": "สั้นๆ"}}"""
    for attempt in range(5):
        try:
            r = or_client.chat.completions.create(
                model=GEMINI_MODEL,
                messages=[{
                    'role': 'user',
                    'content': [
                        {'type': 'text', 'text': prompt_text},
                        {'type': 'input_audio',
                         'input_audio': {'data': audio_b64, 'format': 'mp3'}}
                    ]
                }],
                response_format={'type': 'json_object'},
                temperature=0, max_tokens=100,
            )
            content = r.choices[0].message.content
            m = re.search(r'\{.*\}', content, re.DOTALL)
            if m:
                o = json.loads(m.group()); o['status'] = 'ok'; return o
            return {'mismatch': None, 'reason': 'no json', 'status': 'parse_error'}
        except Exception as e:
            if attempt == 4:
                return {'mismatch': None, 'reason': str(e)[:60], 'status': 'error'}
            time.sleep(2 ** attempt)

_t = max(target, key=lambda d: d['formal_score'])
print('Test:', _t['conv_id'], '→', voice_check(_t['conv_id'], full_text(_t)))

Local audio: 21287
Test: conv_07266_label0 → {'mismatch': True, 'reason': 'รหัส OTP ไม่ตรงกัน', 'status': 'ok'}


In [ ]:
# === Section 7: Rank top-N formal → Gemini voice check → % mismatch ===
kept = [d for d in target if d['action'] != 'remove' and d['realistic'] >= REALISTIC_MIN]
picks = [d for d in kept if d['formal_score'] >= FORMAL_MIN]
print(f'เช็ค {len(picks)} ตัว (formal_score>={FORMAL_MIN}) ~{len(picks)*0.6/GEMINI_WORKERS/60:.0f} นาที')

CKPT_V = f'{OUT_DIR}/voice_ckpt_{MODE}.jsonl'
vdone = {}
if os.path.exists(CKPT_V):
    for ln in open(CKPT_V, encoding='utf-8'):
        if ln.strip():
            o = json.loads(ln); vdone[o['conv_id']] = o

# ⭐ retry: ยังไม่ทำ หรือ status=error/parse_error → ทำใหม่
todo = [d for d in picks
        if d['conv_id'] not in vdone
        or vdone[d['conv_id']].get('status') in ('error','parse_error')]
print(f'top {len(picks)} | สำเร็จแล้ว {len(picks)-len(todo)} | todo {len(todo)} '
      f'~{len(todo)/GEMINI_WORKERS*7/60:.0f} นาที')

lock = threading.Lock()
vf = open(CKPT_V, 'a', encoding='utf-8')
def vwork(d):
    v = voice_check(d['conv_id'], full_text(d))
    rec = {'conv_id':d['conv_id'], 'formal_score':d['formal_score'],
           'mismatch':v.get('mismatch'), 'reason':v.get('reason',''),
           'status':v.get('status','ok')}
    with lock:
        vf.write(json.dumps(rec, ensure_ascii=False) + '\n'); vf.flush()
    return rec
with cf.ThreadPoolExecutor(GEMINI_WORKERS) as ex:
    list(tqdm(ex.map(vwork, todo), total=len(todo), desc='Gemini'))
vf.close()

# รวมผล (เอาเรคคอร์ดล่าสุดของแต่ละ conv_id)
vdone = {}
for ln in open(CKPT_V, encoding='utf-8'):
    if ln.strip():
        o = json.loads(ln); vdone[o['conv_id']] = o
checked = [vdone[d['conv_id']] for d in picks if d['conv_id'] in vdone]

# ⭐ แยก 3 สถานะ — error ไม่นับใน % mismatch
ok    = [r for r in checked if r.get('status')=='ok' and r.get('mismatch') is False]
mm    = [r for r in checked if r.get('status')=='ok' and r.get('mismatch') is True]
err   = [r for r in checked if r.get('status') in ('error','parse_error','skip')]
valid = len(ok) + len(mm)
pct   = len(mm)/max(valid,1)*100
print(f'\n=== ผลตรวจเสียง ===')
print(f'  ✅ เสียงเข้าบริบท: {len(ok)}')
print(f'  🚩 mismatch     : {len(mm)}')
print(f'  ⚠️ error/skip   : {len(err)}  ← รัน Section 7 ซ้ำจะ retry')
print(f'  📊 % mismatch   : {len(mm)}/{valid} = {pct:.2f}%  (ไม่นับ error)')
if mm:
    print('\n🚩 ตัวที่เจอ mismatch:')
    for r in mm[:15]:
        print(f"   {r['conv_id']} (formal={r['formal_score']}): {r.get('reason','')}")
if err:
    print('\n⚠️ ตัวอย่าง error:')
    for r in err[:5]:
        print(f"   {r['conv_id']}: {r.get('reason','')[:50]}")

เช็ค 9652 ตัว (formal_score>=5) ~5 นาที
top 9652 | สำเร็จแล้ว 0 | todo 9652 ~56 นาที


Gemini:   0%|          | 0/9652 [00:00<?, ?it/s]


=== ผลตรวจเสียง ===
  ✅ เสียงเข้าบริบท: 3150
  🚩 mismatch     : 6487
  ⚠️ error/skip   : 15  ← รัน Section 7 ซ้ำจะ retry
  📊 % mismatch   : 6487/9637 = 67.31%  (ไม่นับ error)

🚩 ตัวที่เจอ mismatch:
   conv_00006_label0 (formal=7): น้ำเสียงผู้ชายดูร่าเริงเกินไป ไม่เข้ากับเนื้อหาที่ค่อนข้างเป็นทางการ
   conv_00007_label1 (formal=9): น้ำเสียงผู้ชายไม่เข้ากับเนื้อหา
   conv_00014_label0 (formal=5): น้ำเสียงไม่สุภาพและไม่น่าเชื่อถือ มีคำพูดที่ไม่เป็นทางการ เช่น "ป่ะ" และ "จ้า" ซึ่งไม่เหมาะสมกับบริบทที่เป็นทางการ
   conv_00018_label1 (formal=5): น้ำเสียงไม่น่าเชื่อถือและไม่เป็นทางการ
   conv_00019_label1 (formal=9): น้ำเสียงผู้ชายดูไม่น่าเชื่อถือและไม่เป็นทางการเท่าที่ควร
   conv_00022_label0 (formal=6): น้ำเสียงไม่สุภาพและไม่น่าเชื่อถือ
   conv_00023_label0 (formal=7): น้ำเสียงร่าเริงเกินไป ไม่เข้ากับเนื้อหาที่เป็นทางการ
   conv_00025_label0 (formal=9): ตัวเลขในเสียงกับใน transcript ไม่ตรงกัน
   conv_00030_label1 (formal=8): น้ำเสียงผู้ชายไม่น่าเชื่อถือและไม่เป็นทางการ
   conv_00032_label0

In [ ]:
# === Section 8: Report  ===
report = {
    'mode': MODE,
    'n_total': len(target),
    'relabel': sum(d['action']=='relabel' for d in target),
    'relabel_0to1': sum(d['action']=='relabel' and d['label']==0 and d['gpt_label']==1 for d in target),
    'relabel_1to0': sum(d['action']=='relabel' and d['label']==1 and d['gpt_label']==0 for d in target),
    'remove_broken': sum(d['action']=='remove' for d in target),
    'remove_unrealistic': sum(d['action']!='remove' and d['realistic']<REALISTIC_MIN for d in target),
    'voice_checked_total': len(checked),
    'voice_valid': valid,
    'voice_errors': len(err),
    'voice_mismatch': len(mm),
    'voice_mismatch_pct': round(pct, 2),
}
with open(f'{OUT_DIR}/mismatch_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))

ml = report['relabel']/report['n_total']*100
print(f"\n📊 label แก้ {ml:.1f}% → {'retrain คุ้ม' if ml>5 else 'future work'}")
print(f"📊 voice mismatch {pct:.2f}% → "
      f"{'เสียงสอดคล้องบริบทดี' if pct<3 else 'มี mismatch ต้องดู'}")

{
  "mode": "all",
  "n_total": 21352,
  "relabel": 1085,
  "relabel_0to1": 993,
  "relabel_1to0": 34,
  "remove_broken": 1844,
  "remove_unrealistic": 8,
  "voice_checked_total": 9652,
  "voice_valid": 9637,
  "voice_errors": 15,
  "voice_mismatch": 6487,
  "voice_mismatch_pct": 67.31
}

📊 label แก้ 5.1% → retrain คุ้ม
📊 voice mismatch 67.31% → มี mismatch ต้องดู


In [ ]:
# === Section 9: ฟังเสียง mismatch ที่เจอ (ยืนยันด้วยหู) ===
import IPython.display as ipd

show = mm[:10] if mm else ok[:5]
print('🔊 ฟังตัวอย่าง — เทียบว่า Gemini ตัดสินตรงที่ได้ยินไหม\n')
for r in show:
    cid = r['conv_id']
    d = next((x for x in target if x['conv_id']==cid), None)
    if r.get('mismatch') is True:    flag = '🚩 mismatch'
    elif r.get('mismatch') is False: flag = '✅ ok'
    else:                            flag = '⚠️ error (ไม่ได้ตัดสิน)'
    print(f"{flag} {cid} (formal={r['formal_score']}) — {r.get('reason','')[:60]}")
    if d: print(f'   📝 {full_text(d)[:140]}')
    fn = f'{EXT_AUDIO}/{cid}.mp3'
    if os.path.exists(fn):
        ipd.display(ipd.Audio(fn))

🔊 ฟังตัวอย่าง — เทียบว่า Gemini ตัดสินตรงที่ได้ยินไหม

🚩 mismatch conv_00006_label0 (formal=7) — น้ำเสียงผู้ชายดูร่าเริงเกินไป ไม่เข้ากับเนื้อหาที่ค่อนข้างเป
   📝 เห้ สวัสดีครับ คุณสมศักดิ์ ทางธนาคารโทรมาแจ้งนะครับ ว่าบัตรเครดิตของคุณจะหมดอายุเดือนหน้าอ่ะ อ๋อ ใช่ๆ ครับ แล้วต้องทำไงบ้างหรอ


🚩 mismatch conv_00007_label1 (formal=9) — น้ำเสียงผู้ชายไม่เข้ากับเนื้อหา
   📝 สวัสดีค่ะ คุณลูกค้า ทางธนาคารต้องการยืนยันตัวตนค่ะ กรุณาแจ้ง OTP 6 หลักของคุณให้หน่อยนะคะ อ๋อ ครับ นี่ 123456 ใช่ไหมคะ


🚩 mismatch conv_00014_label0 (formal=5) — น้ำเสียงไม่สุภาพและไม่น่าเชื่อถือ มีคำพูดที่ไม่เป็นทางการ เช
   📝 สวัสดีครับ, แกอยู่ออฟฟิศป่ะ อือ, อยู่จ้า มีอะไรหรือเปล่า จะนัดประชุมวันศุกร์นะ เวลาบ่ายสองอ่ะ


🚩 mismatch conv_00018_label1 (formal=5) — น้ำเสียงไม่น่าเชื่อถือและไม่เป็นทางการ
   📝 ฮัลโหลค่ะ คุณผู้ใช้คือมีปัญหานิดหน่อยเกี่ยวกับบัญชีของคุณนะคะ เอ๊ะ อะไรนะคะ คือมีไวรัสที่ทำให้บัญชีของคุณอาจโดนแฮ็กค่ะ เราต้องการให้คุณทำตาม


🚩 mismatch conv_00019_label1 (formal=9) — น้ำเสียงผู้ชายดูไม่น่าเชื่อถือและไม่เป็นทางการเท่าที่ควร
   📝 หวัดดีครับ คุณผู้โชคดีครับ ผมจากกรมสอบสวนคดีพิเศษนะครับ อ๋อ ครับ คือบัญชีของคุณมีความเกี่ยวข้องกับคดีอาชญากรรมครับ ต้องตรวจสอบ จริงเหรอ ใช่ค


🚩 mismatch conv_00022_label0 (formal=6) — น้ำเสียงไม่สุภาพและไม่น่าเชื่อถือ
   📝 ไงเว้ย คุณสมศักดิ์ ใช่ไหมครับ สอบถามเรื่องนัดหมายหน่อยนะครับ เออ ใช่ครับ มีอะไรหรือครับ เราต้องการ confirm นัดหมออาทิตย์หน้า OK หรือเปล่าครั


🚩 mismatch conv_00023_label0 (formal=7) — น้ำเสียงร่าเริงเกินไป ไม่เข้ากับเนื้อหาที่เป็นทางการ
   📝 สวัสดีครับ คุณลูกค้า ผมชื่อสมชาย จากบริษัทประกันภัยนะครับ อ๋อ สวัสดีครับ เราได้โปรโมชั่นใหม่สำหรับประกันสุขภาพครับ เริ่มต้นเพียง 1,200 บาทต่


🚩 mismatch conv_00025_label0 (formal=9) — ตัวเลขในเสียงกับใน transcript ไม่ตรงกัน
   📝 นี่คือรหัส OTP ของคุณ 123456 กรุณาอย่าเปิดเผยให้ใครทราบ


🚩 mismatch conv_00030_label1 (formal=8) — น้ำเสียงผู้ชายไม่น่าเชื่อถือและไม่เป็นทางการ
   📝 ขอโทษครับ ผมตำรวจนะครับ มีเรื่องด่วนเกี่ยวกับคุณเลย มีอะไรเร็วๆ คุณมีส่วนเกี่ยวข้องกับคดีอะไรสักอย่าง ต้องตรวจสอบเอกสารครับ อ๋อ กำลังขับรถอย


🚩 mismatch conv_00032_label0 (formal=6) — เสียงผู้ชายพูดหมายเลขอ้างอิงไม่ตรงกับใน transcript
   📝 สวัสดีครับ คุณมีกข้อมูลเกี่ยวกับการโอนเงินที่มีหมายเลขอ้างอิง 12345 อยู่ไหมครับ อ่อ มีค่ะ ว่ามาเลย


In [ ]:
# === Preview ผล จาก checkpoint  ===
import re
from collections import Counter

recs = [json.loads(l) for l in open(CKPT, encoding='utf-8') if l.strip()]
# เอาตัวล่าสุดของแต่ละ conv_id (กัน duplicate)
done_preview = {}
for r in recs: done_preview[r['conv_id']] = r
recs = list(done_preview.values())
print(f'✅ done: {len(recs)} / {len(data)} ({len(recs)/len(data)*100:.1f}%)')

acts = Counter(r.get('action','keep') for r in recs)
print('\n=== Action ===')
for a,c in acts.items(): print(f'  {a}: {c} ({c/len(recs)*100:.1f}%)')

get_lbl = lambda cid: int(cid.split('_label')[1][0])
up   = sum(1 for r in recs if r.get('action')=='relabel' and get_lbl(r['conv_id'])==0 and r.get('true_label')==1)
down = sum(1 for r in recs if r.get('action')=='relabel' and get_lbl(r['conv_id'])==1 and r.get('true_label')==0)
print(f'  relabel 0→1: {up}  |  1→0: {down}  ← down ควรน้อยมาก')

# โชว์ relabel 20 ตัว ดูว่า GPT ตัดสินถูกตา
data_dict = {f'conv_{i+1:05d}_label{d["label"]}': d for i,d in enumerate(data)}
print('\n=== ตัวอย่าง RELABEL 20 ตัว — ดูว่าถูกตาไหม ===')
relabels = [r for r in recs if r.get('action')=='relabel'][:100]
for r in relabels:
    cid = r['conv_id']; orig = get_lbl(cid); new = r.get('true_label')
    d = data_dict.get(cid)
    if d:
        text = ' '.join(t['text'] for t in d['turns'])[:120]
        print(f"[{cid}] {orig}→{new} | {r.get('reason','')[:50]}")
        print(f'   {text}')

✅ done: 6111 / 21352 (28.6%)

=== Action ===
  keep: 3569 (58.4%)
  remove: 386 (6.3%)
  relabel: 219 (3.6%)
  error: 1937 (31.7%)
  relabel 0→1: 196  |  1→0: 12  ← down ควรน้อยมาก

=== ตัวอย่าง RELABEL 20 ตัว — ดูว่าถูกตาไหม ===
[conv_00035_label0] 0→1 | ขอ OTP จากผู้ใช้ ซึ่งเป็นลักษณะ scam
   พี่จ๋า สวัสดีครับ นี่ธนาคารนะครับ ค่ะ มีอะไรเหรอคะ เราได้รับคำขอทำรายการแล้วนะครับ รบกวนให้ OTP ที่ส่งไปทาง SMS ด้วยครับ
[conv_00054_label0] 0→1 | เพื่อนขอเงิน 500 บาท ถือเป็น scam classic
   น้องครับ สบายดีไหม นี่, มันคือ อ้น เองนะ ได้เบอร์ใหม่แล้ว บังเอิญว่าไปทำโทรศัพท์หล่น ตอนนี้ไม่มีเงินสด มีสิ่งที่ต้องจ่าย
[conv_00124_label0] 0→1 | ขอ OTP เพื่อยืนยันรายการชัดเจนว่าเป็น scam
   ขอโทษครับ คุณ ชื่อ ดิฉันจากธนาคารนะครับ ต้องการยืนยันการทำรายการแบบว่า refernce number 123456 ครับ. อ๋อ ครับ. ขอ OTP ที่
[conv_00160_label0] 0→1 | ขอให้เหยื่อยืนยัน OTP ชัดเจน
   ขออนุญาตครับ คุณลูกค้า นี่ธนาคารนะครับ เรามีธุรกรรมที่ต้องการยืนยันครับ หมายเลขอ้างอิงคือ 123456789 ครับ อ๋อ ครับ กรุณาย
[conv_00168_label0]